In [1]:
import time

from sklearn.decomposition import TruncatedSVD
import scipy.sparse as sp

A = sp.load_npz("../data/processed/train_matrix.npz")

print("ML2: Truncated SVD across k values")
for k in [10, 20, 50, 100, 200]:
    if k < min(A.shape):
        t = time.time()
        svd_k = TruncatedSVD(n_components=k, random_state=42).fit(A)
        print(f"k = {k:3d} | Fit Time: {time.time() - t }| Explained Variance: {svd_k.explained_variance_ratio_.sum()}")



ML2: Truncated SVD across k values
k =  10 | Fit Time: 0.04562091827392578| Explained Variance: 0.41450971733519104
k =  20 | Fit Time: 0.1284639835357666| Explained Variance: 0.5157783696926869
k =  50 | Fit Time: 0.19843578338623047| Explained Variance: 0.6498148651007624
k = 100 | Fit Time: 0.30773186683654785| Explained Variance: 0.7476915595108335
k = 200 | Fit Time: 0.5395772457122803| Explained Variance: 0.8365700966055098


In [2]:
from sklearn.preprocessing import normalize

t = time.time()
svd = TruncatedSVD(n_components=50, random_state=42).fit(A)
print("fit seconds", time.time() - t)
P_vec = normalize(svd.components_.T) 

fit seconds 0.22819209098815918


In [3]:
import numpy as np
print("ML3")
def similar(j, n=10):
 s = P_vec @ P_vec[j]
 s[j] = -np.inf
 return np.argsort(-s)[:n]

print("Top 10 similar to product 0:", similar(0))


ML3
Top 10 similar to product 0: [1182 3714 3294 1710 3424 1358 2863 1436 3073 3967]


In [4]:
import pandas as pd
train = pd.read_csv("../data/processed/train.csv")

top_10_products=train["StockCode"].value_counts().head(10)

print(top_10_products)

StockCode
85123A    4773
22423     3090
85099B    2985
21212     2446
20725     2432
84879     2423
47566     1975
22383     1894
21232     1884
20727     1860
Name: count, dtype: int64


In [5]:
import pandas as pd
import pickle

print("ML4")
A=sp.load_npz("../data/processed/train_matrix.npz")

with open("../data/processed/cust_map.pkl", "rb") as f:
    cust_map = pickle.load(f)

with open("../data/processed/prod_map.pkl", "rb") as f:
    prod_map = pickle.load(f)

test = pd.read_csv("../data/processed/test.csv")

ML4


In [6]:
t_fit = time.time()
svd = TruncatedSVD(n_components=50, random_state=42).fit(A)
P_vec = normalize(svd.components_.T)
print("fit seconds", time.time() - t_fit)

t_start = time.time()

fit seconds 0.12828683853149414


In [7]:
top_10_products=train["StockCode"].value_counts().head(10).index.tolist()
print(top_10_products)

['85123A', '22423', '85099B', '21212', '20725', '84879', '47566', '22383', '21232', '20727']


In [8]:
baseline = [prod_map[int(i)] for i in top_10_products if str(i).isdigit() and int(i) in prod_map]

print(baseline)

[3286, 184, 144, 15, 717, 2921, 4, 547]


In [9]:
t_start = time.time()

hits_svd = 0
hits_baseline = 0
eval_count = 0

test_customers = test["Customer ID"].unique()

for cust in test_customers:
    if cust not in cust_map:
        continue
    
    cust_idx = cust_map[cust]
    
    actual_products = test[test["Customer ID"] == cust]["StockCode"].unique()
    actual_indices = {prod_map[int(p)] for p in actual_products if str(p).isdigit() and int(p) in prod_map}
    
    
    # SVD Recommendations
    cust_vector = A.getrow(cust_idx).toarray().ravel()
    scores = A.getrow(cust_idx) @ P_vec @ P_vec.T
    scores = scores.ravel()
    scores[cust_vector > 0] = -np.inf
    svd_recs = np.argsort(-scores)[:10]
    
    if any(idx in actual_indices for idx in svd_recs):
        hits_svd += 1
    if any(idx in actual_indices for idx in baseline):
        hits_baseline += 1
        
    eval_count += 1

elapsed = time.time() - t_start

print("Time: {:.2f}s".format(elapsed))
print(f"SVD Hit Rate @ 10: {hits_svd / eval_count}")
print(f"Baseline Hit Rate @ 10: {hits_baseline / eval_count}")

Time: 2.09s
SVD Hit Rate @ 10: 0.1802154750244858
Baseline Hit Rate @ 10: 0.3751224289911851


In [13]:
from sklearn.decomposition import PCA

print("ML5: PCA")

start = time.time()

pca = PCA(n_components=50, svd_solver='arpack', random_state=42).fit(A)
pca_time = time.time() - start


print(f"Fit Time: {pca_time} and Explained Variance: {pca.explained_variance_ratio_.sum()}")

ML5: PCA
Fit Time: 0.14235210418701172 and Explained Variance: 0.6501560362177455


In [11]:
print("PCA vs SVD")

print("Model Comparison Table (k = 50)")
print ("PCA Variance", pca.explained_variance_ratio_.sum())
print ("SVD Variance", svd.explained_variance_ratio_.sum())

PCA vs SVD
Model Comparison Table (k = 50)
PCA Variance 0.6501560362177455
SVD Variance 0.6498148651007624


In [21]:
from sklearn.cluster import KMeans

print("ML 6")

#reduce dimensions

customer_projected_2 = svd.transform(A)[:, :2]

#cluster
kmeans = KMeans(n_clusters=4, random_state=42)

cluster_labels= kmeans.fit_predict(customer_projected_2)

cluster_dict = {i: label for i, label in enumerate(cluster_labels)}

train["Cluster"] = train["Customer ID"].map(cust_map).map(cluster_dict)

for i in range(4):
    cluster_info = train[train["Cluster"] == i]
    top_3_products = cluster_info["StockCode"].value_counts().head(3).index.tolist()
    top_2_countries = cluster_info["Country"].value_counts().head(2).index.tolist()

    print(f"Cluster {i}:")
    print(f"Top 3 Products: {top_3_products}")
    print(f"Top 2 Countries: {top_2_countries}")
    print()


ML 6
Cluster 0:
Top 3 Products: ['85123A', '22423', '84879']
Top 2 Countries: ['United Kingdom', 'Germany']

Cluster 1:
Top 3 Products: ['82494L', '82486', '82482']
Top 2 Countries: ['United Kingdom']

Cluster 2:
Top 3 Products: ['85123A', '20725', '85099B']
Top 2 Countries: ['United Kingdom', 'EIRE']

Cluster 3:
Top 3 Products: ['85099B', '85123A', '20725']
Top 2 Countries: ['United Kingdom', 'France']

